select TO_CHAR(rental_date, 'YYYYY-MM'), customer_id from rental limit 25

select extract(month  from rental_date)

In [58]:
import psycopg2
import pandas as pd


conn = psycopg2.connect(
    host="localhost", database="dvdrental", user="postgres", password="postgres"
)
df = pd.read_sql_query("SELECT * FROM rental", conn)
df["rental_date"] = pd.to_datetime(df["rental_date"])

df["rental_month"] = df["rental_date"].dt.to_period("M")
df = df.groupby(["rental_month", "customer_id"]).size().reset_index(name="count")
df["next_month"] = df["rental_month"] + 1
df2 = pd.merge(
    df[["customer_id", "rental_month", "count"]],
    df[["customer_id", "next_month", "count"]],
    how="outer",
    left_on=["customer_id", "rental_month"],
    right_on=["customer_id", "next_month"],
    suffixes=("_this", "_perv"),
)

C:\Users\mamma\AppData\Local\Temp\ipykernel_35836\3429935446.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("SELECT * FROM rental", conn)


In [59]:
df2

,customer_id,rental_month,count_this,next_month,count_perv
0,1,2005-05,2.0,NaT,NaN
1,1,2005-06,7.0,2005-06,2.0
2,1,2005-07,12.0,2005-07,7.0
3,1,2005-08,11.0,2005-08,12.0
4,1,NaT,NaN,2005-09,11.0
...,...,...,...,...,...
3226,599,2005-05,1.0,NaT,NaN
3227,599,2005-06,4.0,2005-06,1.0
3228,599,2005-07,7.0,2005-07,4.0
3229,599,2005-08,7.0,2005-08,7.0


In [63]:
df2[df2["count_this"].notna() & df2["count_perv"].notna()].groupby(["customer_id", "rental_month"]).size().reset_index(name="count")

,customer_id,rental_month,count
0,1,2005-06,1
1,1,2005-07,1
2,1,2005-08,1
3,2,2005-06,1
4,2,2005-07,1
...,...,...,...
1696,598,2005-07,1
1697,598,2005-08,1
1698,599,2005-06,1
1699,599,2005-07,1


In [85]:
active  =    df2[df2["count_this"].notna()].groupby("rental_month")["customer_id"].nunique().reset_index(name="active_customers_last_month")
retained =    df2[df2["count_this"].notna() & df2["count_perv"].notna()].groupby(["customer_id", "rental_month"]).size().reset_index(name="count").groupby("rental_month").size().reset_index(name="retained_customers")

In [86]:
active["rental_month"] = active["rental_month"] + 1

In [87]:
active

,rental_month,active_customers_last_month
0,2005-06,520
1,2005-07,590
2,2005-08,599
3,2005-09,599
4,2006-03,158


In [103]:
df2["cum_count"] = df2.groupby(["customer_id", "rental_month"])[["count_this"]].cumsum()
# But .cumsum() is a transform, not an aggregation — so it keeps row-level alignment.

# Transform-like functions (cumsum, rank, ffill, bfill, etc.) → preserve original index.

# Aggregation functions (sum, count, mean, etc.) → return group-level index (MultiIndex if >1 grouping key).

In [104]:
df2

,customer_id,rental_month,count_this,next_month,count_perv,cum_count
0,1,2005-05,2.0,NaT,NaN,2.0
1,1,2005-06,7.0,2005-06,2.0,7.0
2,1,2005-07,12.0,2005-07,7.0,12.0
3,1,2005-08,11.0,2005-08,12.0,11.0
4,1,NaT,NaN,2005-09,11.0,NaN
...,...,...,...,...,...,...
3226,599,2005-05,1.0,NaT,NaN,1.0
3227,599,2005-06,4.0,2005-06,1.0,4.0
3228,599,2005-07,7.0,2005-07,4.0,7.0
3229,599,2005-08,7.0,2005-08,7.0,7.0


In [120]:
def longest_consecutive(nums):
  nums.sort()
  print(nums)
  i = 0
  j = 0
  largest = 0
  while i<len(nums):
    j=i+1
    while (j<len(nums)) and (nums[j]==nums[j-1]+1):
      j+=1
    largest = max(largest, j-i)
    i=j

  return largest
longest_consecutive( [9,1,4,7,3,-1,0,5,8,-1,6])

[-1, -1, 0, 1, 3, 4, 5, 6, 7, 8, 9]


7

In [88]:
retention = active.merge(retained, on="rental_month", how="left").fillna(0)
retention["retention_rate"] = (
    retention["retained_customers"] / retention["active_customers_last_month"]
).round(3)
retention

,rental_month,active_customers_last_month,retained_customers,retention_rate
0,2005-06,520,512.0,0.985
1,2005-07,590,590.0,1.000
2,2005-08,599,599.0,1.000
3,2005-09,599,0.0,0.000
4,2006-03,158,0.0,0.000


In [ ]:
        new_node = Node(data)
        if not self.head:
            self.head = new_node
            return
        last = self.head
        while last.next:
            last = last.next
        last.next = new_node
        

In [121]:
class Node:
    def __init__(self, data):
        self.data= data
        self.next = None
        self.prev = None

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, data):
        new_node = Node(data)
        if not self.head:
            self.head = new_node
            return
        last_node = self.head
        while last_node.next:
            last_node = last_node.next
        last_node.next = new_node
        new_node.prev = last_node

    def prepend(self, data):
        new_node = Node(data)
        if not self.head:
            self.head = new_node
            return
        new_node.next = self.head
        self.head.prev = new_node
        self.head = new_node

    def display(self):
        current_node = self.head
        while current_node:
            print(current_node.data, end=" -> ")
            current_node = current_node.next
    
ls = LinkedList()
ls.append(1)
ls.append(2)
ls.append(3)
ls.prepend(0)
ls.display()
print()

0 -> 1 -> 2 -> 3 -> 


In [ ]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next = None

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, data):
        # ... implementation for appending a node
        pass

    def delete(self, data):
        # ... implementation for deleting a node
        pass

In [130]:
d = {'banana': 3, 'apple': 2, 'pear': 1, 'orange': 4}
# sorted(d.items(), key=lambda x: x[1], reverse=False)

sorted(d.items())
sorted(d.items(), key = lambda item:item[1])

[('pear', 1), ('apple', 2), ('banana', 3), ('orange', 4)]

In [133]:
def append_to_list(val, my_list=[]):
    my_list.append(val)
    return my_list
print(append_to_list(1))  # [1]
print(append_to_list(2))  # ???
print(append_to_list(2))

[1]
[1, 2]
[1, 2, 2]


In [134]:
print(append_to_list(2))

[1, 2, 2, 2]


In [139]:
# user interactions
df = pd.DataFrame({
    'user_id': [1, 1, 2, 2, 3],
    'event': ['click', 'view', 'click', 'click', 'view'],
    'timestamp': pd.to_datetime(['2023-01-01 10:00', '2023-01-01 10:05', '2023-01-02 11:00', '2023-01-02 11:05', '2023-01-03 12:00'])
})
# How would you get the first and last interaction timestamp for each user?
df.groupby('user_id').agg({'timestamp': ['min', 'max']}).reset_index()

user_id           timestamp                    
                          min                 max
0       1 2023-01-01 10:00:00 2023-01-01 10:05:00
1       2 2023-01-02 11:00:00 2023-01-02 11:05:00
2       3 2023-01-03 12:00:00 2023-01-03 12:00:00

Create a pivot table showing the average revenue per user per day from a transactional DataFrame.

In [144]:
import psycopg2
conn = psycopg2.connect(
    host="localhost", database="dvdrental", user="postgres", password="postgres"
)
df_r = pd.read_sql_query("SELECT * FROM rental", conn)
df_r["rental_date"] = pd.to_datetime(df["rental_date"])

df_p = pd.read_sql_query("SELECT * FROM payment", conn)


C:\Users\mamma\AppData\Local\Temp\ipykernel_35836\187672093.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_r = pd.read_sql_query("SELECT * FROM rental", conn)
C:\Users\mamma\AppData\Local\Temp\ipykernel_35836\187672093.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_p = pd.read_sql_query("SELECT * FROM payment", conn)


In [149]:
df = pd.merge(df_p, df_r, on="rental_id", suffixes=["","r"], how="inner")

In [153]:
df_p

,payment_id,customer_id,staff_id,rental_id,amount,payment_date
0,17503,341,2,1520,7.99,2007-02-15 22:25:46.996577
1,17504,341,1,1778,1.99,2007-02-16 17:23:14.996577
2,17505,341,1,1849,7.99,2007-02-16 22:41:45.996577
3,17506,341,2,2829,2.99,2007-02-19 19:39:56.996577
4,17507,341,2,3130,7.99,2007-02-20 17:31:48.996577
...,...,...,...,...,...,...
14591,32094,245,2,12682,2.99,2007-05-14 13:44:29.996577
14592,32095,251,1,14107,0.99,2007-05-14 13:44:29.996577
14593,32096,252,2,13756,4.99,2007-05-14 13:44:29.996577
14594,32097,263,1,15293,0.99,2007-05-14 13:44:29.996577


In [155]:
df.pivot_table(index=["customer_id", "rental_date"], values="amount", aggfunc="sum").unstack("customer_id")

amount                                          ...        \
customer_id            1   2   3   4     5   6   7   8     9   10   ...   590   
rental_date                                                         ...         
2005-06-14 22:53:33    NaN NaN NaN NaN   NaN NaN NaN NaN   NaN NaN  ...   NaN   
2005-06-14 22:55:13    NaN NaN NaN NaN   NaN NaN NaN NaN   NaN NaN  ...   NaN   
2005-06-14 23:00:34    NaN NaN NaN NaN   NaN NaN NaN NaN   NaN NaN  ...   NaN   
2005-06-14 23:12:46    NaN NaN NaN NaN   NaN NaN NaN NaN   NaN NaN  ...   NaN   
2005-06-14 23:16:26    NaN NaN NaN NaN   NaN NaN NaN NaN   NaN NaN  ...   NaN   
...                    ...  ..  ..  ..   ...  ..  ..  ..   ...  ..  ...   ...   
2005-08-23 22:26:47    NaN NaN NaN NaN   NaN NaN NaN NaN   NaN NaN  ...   NaN   
2005-08-23 22:42:48    NaN NaN NaN NaN   NaN NaN NaN NaN   NaN NaN  ...   NaN   
2005-08-23 22:43:07    NaN NaN NaN NaN   NaN NaN NaN NaN   NaN NaN  ...   NaN   
2005-08-23 22:50:12    NaN NaN NaN NaN   NaN NaN NaN NaN   NaN NaN  ...   NaN   
2006-02-14 15:16:03    NaN NaN NaN NaN  0.99 NaN NaN NaN  4.99 NaN  ...  2.99   

                                                               
customer_id         591   592 593 594 595   596   597 598 599  
rental_date                                                    
2005-06-14 22:53:33 NaN   NaN NaN NaN NaN   NaN   NaN NaN NaN  
2005-06-14 22:55:13 NaN   NaN NaN NaN NaN   NaN   NaN NaN NaN  
2005-06-14 23:00:34 NaN   NaN NaN NaN NaN   NaN   NaN NaN NaN  
2005-06-14 23:12:46 NaN  6.99 NaN NaN NaN   NaN   NaN NaN NaN  
2005-06-14 23:16:26 NaN   NaN NaN NaN NaN   NaN   NaN NaN NaN  
...                  ..   ...  ..  ..  ..   ...   ...  ..  ..  
2005-08-23 22:26:47 NaN   NaN NaN NaN NaN   NaN   NaN NaN NaN  
2005-08-23 22:42:48 NaN   NaN NaN NaN NaN   NaN   NaN NaN NaN  
2005-08-23 22:43:07 NaN   NaN NaN NaN NaN   NaN   NaN NaN NaN  
2005-08-23 22:50:12 NaN   NaN NaN NaN NaN   NaN   NaN NaN NaN  
2006-02-14 15:16:03 NaN  0.99 NaN NaN NaN  0.99  4.99 NaN NaN  

[14364 rows x 599 columns]

# Numpy

In [236]:
import numpy as np
a1 = np.array([1, 2, 3])
a2= np.random.random([3,3])
a3 = np.arange(9).reshape(3, 3)
a4  = np.full((3, 3), 5)
a5 = np.array([1.0, 2.0, np.nan, np.nan, 4.0, 5.0])
a6 = np.arange(81).reshape(3,3,3,3)


In [245]:
a6

array([[[[ 0,  1,  2],
         [ 3,  4,  5],
         [ 6,  7,  8]],

        [[ 9, 10, 11],
         [12, 13, 14],
         [15, 16, 17]],

        [[18, 19, 20],
         [21, 22, 23],
         [24, 25, 26]]],


       [[[27, 28, 29],
         [30, 31, 32],
         [33, 34, 35]],

        [[36, 37, 38],
         [39, 40, 41],
         [42, 43, 44]],

        [[45, 46, 47],
         [48, 49, 50],
         [51, 52, 53]]],


       [[[54, 55, 56],
         [57, 58, 59],
         [60, 61, 62]],

        [[63, 64, 65],
         [66, 67, 68],
         [69, 70, 71]],

        [[72, 73, 74],
         [75, 76, 77],
         [78, 79, 80]]]])

In [246]:
a6[0, 1, 1:3] 

array([[12, 13, 14],
       [15, 16, 17]])

In [219]:
print(a5)
# Access only non-missing values
np.ma.masked_invalid(a5).compressed()

[ 1.  2. nan nan  4.  5.]


array([1., 2., 4., 5.])

In [214]:
a1[::-1]

array([3, 2, 1])

In [207]:
np.linalg.det(a2)

-0.06602413787050694

In [210]:
a3.flatten()

array([0, 1, 2, 3, 4, 5, 6, 7, 8])

In [211]:
a3.ravel()

array([0, 1, 2, 3, 4, 5, 6, 7, 8])

In [203]:
np.array([[2.0] * 4] * 3).astype(np.int16)

array([[2, 2, 2, 2],
       [2, 2, 2, 2],
       [2, 2, 2, 2]], dtype=int16)

In [208]:
np.linalg.norm(a3, ord=None, axis=None)


14.2828568570857

In [209]:
np.equal(a1, a2[0])

array([False, False, False])

In [185]:
a2[a2 > 0.5] 

array([0.82134902, 0.53621257, 0.5320545 ])

In [189]:
np.vstack((a1,a3))

array([[1, 2, 3],
       [0, 1, 2],
       [3, 4, 5],
       [6, 7, 8]])

In [197]:
np.concatenate((a2,a3), axis=0)

array([[0.82134902, 0.48912678, 0.05118375],
       [0.01471161, 0.53621257, 0.5320545 ],
       [0.28086432, 0.27112886, 0.02807407],
       [0.        , 1.        , 2.        ],
       [3.        , 4.        , 5.        ],
       [6.        , 7.        , 8.        ]])

In [175]:
a1 , a2, a3, a4

(array([1, 2, 3]),
 array([[0.20543418, 0.68133807, 0.60909172],
        [0.98689223, 0.20137481, 0.69813423],
        [0.7285902 , 0.30681798, 0.13965526]]),
 array([[0, 1, 2],
        [3, 4, 5],
        [6, 7, 8]]),
 array([[5, 5, 5],
        [5, 5, 5],
        [5, 5, 5]]))

In [177]:
a1 @ a2

array([1.6933652 , 2.37493848, 1.19951496])

In [179]:
a3*a4, a3 @ a4

(array([[ 0,  5, 10],
        [15, 20, 25],
        [30, 35, 40]]),
 array([[ 15,  15,  15],
        [ 60,  60,  60],
        [105, 105, 105]]))

In [184]:
np.var(a2, axis=1)

array([0.09948003, 0.05995824, 0.01367481])

In [172]:
a1 * a2

array([[0.09930321, 0.52001792, 1.75707127],
       [0.50201845, 0.28811069, 2.31250698],
       [0.97149153, 1.55533688, 2.21689964]])

In [163]:
a1+a2

array([[1.01988674, 2.37844886, 3.03941841],
       [1.20009313, 2.0530058 , 3.58142295],
       [1.43106742, 2.13762448, 3.58691143]])

# Pandas

In [220]:
import pandas as pd
import numpy as np

df = pd.DataFrame({'category': ['A', 'B', 'C', 'A'],
                   'value': [10, 20, 30, 40]})

df

,category,value
0,A,10
1,B,20
2,C,30
3,A,40


In [221]:
# Using a dictionary to map values
category_map = {'A': 'Group 1', 'B': 'Group 2', 'C': 'Group 3'}
df['category_mapped'] = df['category'].map(category_map)

# Using a function
df['value_doubled'] = df['value'].map(lambda x: x * 2)
df

,category,value,category_mapped,value_doubled
0,A,10,Group 1,20
1,B,20,Group 2,40
2,C,30,Group 3,60
3,A,40,Group 1,80


In [227]:
# It applies a function to every single element (cell) in a DataFrame. It works element-wise.
pd.DataFrame({
    'A': [1, 2, 3],
    'B': [10, 20, 30],
    'C': [100, 200, 300]
}).applymap(lambda x: f'${x:.2f}')

C:\Users\mamma\AppData\Local\Temp\ipykernel_35836\704081592.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  }).applymap(lambda x: f'${x:.2f}')


,A,B,C
0,$1.00,$10.00,$100.00
1,$2.00,$20.00,$200.00
2,$3.00,$30.00,$300.00


In [225]:
# You need to perform a calculation that involves multiple columns in a row (e.g., axis=1). For example, calculating the sum or range of values across several columns for each row.
# You need to perform an operation on an entire column as a whole (e.g., axis=0), like calculating the max() value of each column.
# The logic you want to apply is more complex than a simple element-wise transformation.
pd.DataFrame({
    'A': [1, 2, 3],
    'B': [10, 20, 30],
    'C': [100, 200, 300]
}).apply(np.sum, axis=0)

A      6
B     60
C    600
dtype: int64

In [229]:
df

,category,value,category_mapped,value_doubled
0,A,10,Group 1,20
1,B,20,Group 2,40
2,C,30,Group 3,60
3,A,40,Group 1,80


In [234]:
df.apply(lambda x: [1, 2], axis=1), df.apply(lambda x: [1, 2], axis=1,result_type='expand'), df.apply(lambda x: [1, 2], axis=0)

(0    [1, 2]
 1    [1, 2]
 2    [1, 2]
 3    [1, 2]
 dtype: object,
    0  1
 0  1  2
 1  1  2
 2  1  2
 3  1  2,
    category  value  category_mapped  value_doubled
 0         1      1                1              1
 1         2      2                2              2)

In [232]:
pd.DataFrame({
    'A': [1, 2, 3],
    'B': [10, 20, 30],
    'C': [100, 200, 300]
}).apply(np.sqrt)

,A,B,C
0,1.000000,3.162278,10.000000
1,1.414214,4.472136,14.142136
2,1.732051,5.477226,17.320508


In [235]:
pd.DataFrame({
    'A': [1, 2, 3],
    'B': [10, 20, 30],
    'C': [100, 200, 300]
}).style.format('${:.2f}')


,A,B,C
0,$1.00,$10.00,$100.00
1,$2.00,$20.00,$200.00
2,$3.00,$30.00,$300.00


In [273]:
dates_with_gaps

DatetimeIndex(['2024-01-01', '2024-01-08', '2024-01-22', '2024-01-29',
               '2024-02-12', '2024-02-19', '2024-02-26', '2024-03-04'],
              dtype='datetime64[ns]', freq=None)

In [274]:
dates

DatetimeIndex(['2024-01-01', '2024-01-08', '2024-01-15', '2024-01-22',
               '2024-01-29', '2024-02-05', '2024-02-12', '2024-02-19',
               '2024-02-26', '2024-03-04'],
              dtype='datetime64[ns]', freq='W-MON')

In [260]:
# Example values
countries = ['US', 'UK']
skus = ['A', 'B']
dates = pd.date_range(start='2024-01-01', periods=10, freq='W-MON')  # Weekly Mondays

# Introduce missing weeks manually
dates_with_gaps = dates.delete([2, 5])  # Remove a couple of weeks

# Build cartesian product
records = []
np.random.seed(42)
for country in countries:
    for sku in skus:
        for date in dates_with_gaps:
            value = np.random.randint(10, 100)
            records.append((country, sku, date, value))

# Create DataFrame
df = pd.DataFrame(records, columns=['country', 'sku', 'date', 'value'])

# Set MultiIndex
df.set_index(['country', 'sku', 'date'], inplace=True)

# Sort for readability
df = df.sort_index()

df.head(2)

value
country sku date             
UK      A   2024-01-01     39
            2024-01-08     47

In [267]:
df["week"]=pd.PeriodIndex(df.index.get_level_values(2), freq="W")

In [270]:
complete_range = pd.date_range(start =df.index.get_level_values(2).min() , end=df.index.get_level_values(2).max(), freq='W-MON')

In [272]:
complete_range

DatetimeIndex(['2024-01-01', '2024-01-08', '2024-01-15', '2024-01-22',
               '2024-01-29', '2024-02-05', '2024-02-12', '2024-02-19',
               '2024-02-26', '2024-03-04'],
              dtype='datetime64[ns]', freq='W-MON')

In [271]:
df.index.get_level_values(2).unique()

DatetimeIndex(['2024-01-01', '2024-01-08', '2024-01-22', '2024-01-29',
               '2024-02-12', '2024-02-19', '2024-02-26', '2024-03-04'],
              dtype='datetime64[ns]', name='date', freq=None)

In [280]:
# Step 1: Reset MultiIndex so we can group by country and sku
df_reset = df.reset_index()

# Step 2: Group by country and sku, and apply asfreq with date as index
filled = (
    df_reset
    .set_index('date')
    .groupby(['country', 'sku'])
    .apply(lambda g: g.sort_index().asfreq('W-MON'))[['value']]
)

# Step 3: Reset index and re-set full MultiIndex if needed
# filled = filled.reset_index().set_index(['country', 'sku', 'date']).sort_index()

filled.head(10)

C:\Users\mamma\AppData\Local\Temp\ipykernel_35836\979442176.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sort_index().asfreq('W-MON'))[['value']]


value
country sku date             
UK      A   2024-01-01   39.0
            2024-01-08   47.0
            2024-01-15    NaN
            2024-01-22   11.0
            2024-01-29   73.0
            2024-02-05    NaN
            2024-02-12   69.0
            2024-02-19   30.0
            2024-02-26   42.0
            2024-03-04   85.0